# Hand-crafted multi-goal ranking in the right room

Run in a GPU Colab. This uses the six collection goals and eight square bands from the wide one-macro run. Each band adds legal intersections of a fixed 2-unit lattice centered on each goal. All earlier pairs stay in the pool. Pairs can be from the same band or different bands and must differ by at least 0.5 units in distance to the goal. Within this room, straight-line distance equals shortest legal path distance.

Fit a fresh shared ranking adapter from identity, then warm-start each next band. The encoder, adapter recipe, loss scale, and 500 updates per band match the collection run. This is a dense hand-crafted control with clean distance labels and more spatial coverage. The experiment measures cost geometry on rendered observations.

Checkpoints and per-band reports are saved to Drive. Set `RESUME_RUN_ID` to a printed run ID to reuse completed checkpoints. After fitting, read the table for the change in guidance across each square edge and the heatmaps for the change in geometry.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
COLLECTION_RUN=(DRIVE/'datasets/curriculum_cost_adapter/experience_one_macro_adapter_wide_v1'
                /'20261108T091827483705Z')
RESUME_RUN_ID=None
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/handcrafted_half_square_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-handcrafted-half-square')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='24dfc4324acd9daff5a930c9cf6b1a41e22dc846'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
ARTIFACTS=Path('/content/handcrafted-half-square-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Run ID:',RUN_ID)
print('Output:',OUTPUT)

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in [(ENCODER_DRIVE,ENCODER,ENCODER_SHA)]:
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

## Fit and save the eight checkpoints

The encoder renders and embeds the positions once per session. Each checkpoint is saved as soon as it is trained.

In [ ]:
started=time.monotonic()
shell([sys.executable,'-m','src.mylonas_ijepa_wm.handcrafted_half_square',
       '--collection-run',COLLECTION_RUN,'--output-dir',OUTPUT,'--encoder-checkpoint',ENCODER,
       '--dino-wm-root',DINO_ROOT,'--device','cuda'],cwd=REPO)
print('Run minutes:',round((time.monotonic()-started)/60,2))

## Check whether the band became easier to enter

The same held-out midpoint grid is used for raw MSE and every checkpoint. For each goal and band, the boundary table selects adjacent grid points where the step crosses the band's outer edge toward the goal and reduces physical distance by at least 0.5 units. It reports how often MSE also decreases. The columns compare raw MSE, the incoming checkpoint, and the checkpoint just fitted. A blank percentage means there are no eligible edges, for example when the square already covers the room.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

report=json.loads((OUTPUT/'comparison.json').read_text())
seen=report['seen_goals']
print(f"{report['heldout_points']} fixed held-out midpoint grid positions")
goal_table=pd.DataFrame([
    dict(goal=i,x=g[0],y=g[1],split='fit' if i<seen else 'held out')
    for i,g in enumerate(report['goals'])
])
display(goal_table.round(2))
GOALS_TO_INSPECT=[1, 0, len(report['goals'])-1]  # Collection indices match the trajectory viewer.

def rank_columns(rows):
    return dict(seen_goal_rank_pct=100*np.mean([r['ranking']['correct'] for r in rows[:seen]]),
                heldout_goal_rank_pct=100*np.mean([r['ranking']['correct'] for r in rows[seen:]]))

stage_rows=[dict(band='raw',**rank_columns(report['raw']['metrics']))]
for stage in report['stages']:
    b=stage['band']
    lo,hi=report['bands'][b-1]
    stage_rows.append(dict(band=b,half_width=f'{lo}–{hi}',new_goal_positions=sum(stage['new_points_per_goal']),
        cumulative_pairs=stage['pairs'],train_before=stage['train_before']['rank_accuracy'],
        train_after=stage['train_after']['rank_accuracy'],
        **rank_columns(stage['metrics'])))
display(pd.DataFrame(stage_rows).round(3))

def pct(metric):
    value=metric['correct']
    return None if value is None else 100*value

rows=[]
for i in GOALS_TO_INSPECT:
    for k,stage in enumerate(report['stages']):
        raw=report['raw']['metrics'][i]['boundaries'][k]
        incoming=(report['raw'] if k==0 else report['stages'][k-1])['metrics'][i]['boundaries'][k]
        after=stage['metrics'][i]['boundaries'][k]
        rows.append(dict(goal=i,band=k+1,heldout_edges=after['pairs'],
                         raw_pct=pct(raw),incoming_pct=pct(incoming),after_pct=pct(after)))
boundary=pd.DataFrame(rows)
display(boundary.round(1))
boundary.to_csv(OUTPUT/'selected_goal_boundaries.csv',index=False)

## See what each fit changed

Each goal has three rows of three stages. In each cell, the top map is the current cost, divided by that map's median cost across the room. All top maps for a goal share a color scale. The bottom map shows the change from the preceding checkpoint on that same normalized scale. Blue indicates a relative cost decrease and red a relative increase. A global change in MSE scale alone disappears in this view. Orange lines mark the band being fitted.

For each map, the title also gives the original median latent MSE. Raw MSE is the reference in the first cell. The heatmaps extend to x = 60.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

grid=report['grid']
xs,ys=grid['xs'],grid['ys']
shape=(len(ys),len(xs))
plot_dir=OUTPUT/'heatmaps'
plot_dir.mkdir(exist_ok=True)

for index in GOALS_TO_INSPECT:
    gx,gy=report['goals'][index]
    stages=[report['raw'],*report['stages']]
    costs=np.asarray([stage['costs'][index] for stage in stages]).reshape(-1,*shape)
    medians=np.median(costs,axis=(1,2))
    values=costs/np.maximum(medians[:,None,None],1e-9)
    change=np.diff(values,axis=0,prepend=values[:1])
    max_delta=max(float(np.abs(change).max()),1e-9)
    cols=3
    rows=(len(stages)+cols-1)//cols
    fig=plt.figure(figsize=(12,5.2*rows),layout='constrained')
    layout=fig.add_gridspec(2*rows,cols+1,width_ratios=[1]*cols+[.07])
    cost_bar=fig.add_subplot(layout[:rows,-1])
    change_bar=fig.add_subplot(layout[rows:,-1])
    for j in range(rows*cols):
        row,col=divmod(j,cols)
        top=fig.add_subplot(layout[2*row,col])
        bottom=fig.add_subplot(layout[2*row+1,col])
        if j>=len(stages):
            top.set_visible(False)
            bottom.set_visible(False)
            continue
        current=top.pcolormesh(xs,ys,values[j],shading='nearest',cmap='viridis',
                               vmin=0,vmax=float(values.max()))
        delta=bottom.pcolormesh(xs,ys,change[j],shading='nearest',cmap='RdBu_r',vmin=-max_delta,vmax=max_delta)
        label='Raw MSE' if j==0 else f'Band {j} · {report["bands"][j-1][0]}–{report["bands"][j-1][1]}'
        top.set_title(f'{label} · median {medians[j]:.3g}')
        bottom.set_title('Reference' if j==0 else 'Change from previous')
        for ax in (top,bottom):
            ax.scatter(gx,gy,marker='*',s=62,facecolor='black',edgecolor='white',zorder=5)
            if j:
                for radius in report['bands'][j-1]:
                    if radius:
                        ax.add_patch(Rectangle((gx-radius,gy-radius),2*radius,2*radius,
                            fill=False,edgecolor='#ff9d19',linestyle='--',linewidth=1.1))
            ax.set(xlim=(35,60),ylim=(4,60),aspect='equal',xlabel='Ball x')
            if col==0:
                ax.set_ylabel('Ball y')
    fig.colorbar(current,cax=cost_bar,label='MSE divided by map median')
    fig.colorbar(delta,cax=change_bar,label='Change in normalized MSE')
    fig.suptitle(f'Goal {index} ({gx:.2f}, {gy:.2f}) · {"fit" if index<seen else "held out"}')
    path=plot_dir/f'goal_{index}_changes.png'
    fig.savefig(path,dpi=130,bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(path)))